# Inpatient EDA

Basic EDA on inpatient encounters to understand:
- Cost drivers (diagnosis, specialty, payer, place of service, disposition)
- Readmission patterns (30/60/90 days)
- High-cost and high-utilization cohorts (patient-level)
- Specialty / facility drill-down

Operational definitions:
- **Event / encounter:** one row in the dataset (one inpatient claim/encounter record).
- **Patient:** unique `deid_personid`.
- **LOS (length of stay):** `los_days = (dischargedate - admitdate)` in days, clipped to minimum 1 day.
- **Readmission flags:** `readmission`, `readmission60`, `readmission90` (0/1).
- **High-cost patient:** patient in top 10% or 5% of total cost (p90/p95), depending on the analysis.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
gt_ip = pd.read_csv('ehn-gt-cleaned-inpatient-20251223000.csv')

In [3]:
gt_ip.head()

,admitdate,dischargedate,deid_personid,planpayer,patient_city,patient_state,patient_zip,pcpnpi,pcpname,ehn_vs_non,...,servicing_specialty,rptgrouper,servicefacility,servicefacilitynpi,readmission,readmission60,readmission90,followup7dayind,followup30daytcoc,dischargedisposition
0,2024-12-22,2024-12-29,56a053a2c5f5a16146246c8cd206b15f,Emory Anthem Commercial,BUFORD,GA,30519,UNKNOWN,UNKNOWN,Unknown,...,Clinical Medical Laboratory,IP BEHAVIORAL HEALTH,"SUNRISE DETOX ALPHARETTA, LLC",1427440999,1,1,1,1,218.36,HOME
1,2024-11-30,2024-12-13,282bfaa0946165ec48dd67c64bd43a67,Emory Anthem Commercial,UNION CITY,GA,30291,UNKNOWN,UNKNOWN,Unknown,...,Pathology,IP SURGICAL (ADULT),"PAULDING MEDICAL CENTER, INC.",1457329435,0,0,0,0,"862,755.42",HOME
2,2024-12-04,2024-12-16,72991e18eaf8a5cf2ccad525370c647f,Emory MSSP,LILBURN,GA,30047,1194091025,"GROSE, LEE",EHN,...,Nurse Practitioner,IP SNF,"ENGLAND ASSOCIATES, LP",1144530148,0,0,0,1,"3,814.60",HOME
3,2024-11-30,2024-12-04,72991e18eaf8a5cf2ccad525370c647f,Emory MSSP,LILBURN,GA,30047,1194091025,"GROSE, LEE",EHN,...,Internal Medicine,IP MEDICAL (ADULT),"EASTSIDE MEDICAL CENTER, LLC",1467406132,0,0,0,0,"13,180.36",IP SNF
4,2024-05-13,2024-05-16,503d9c68c71212bda9697f48ee3a5580,Emory Anthem Commercial,VILLA RICA,GA,30180,UNKNOWN,UNKNOWN,Unknown,...,Obstetrics & Gynecology,IP SNF,"TANNER MEDICAL CENTER, INC",1801883780,0,0,0,1,"8,141.20",HOME


In [4]:
gt_ip.columns

Index(['admitdate', 'dischargedate', 'deid_personid', 'planpayer',
       'patient_city', 'patient_state', 'patient_zip', 'pcpnpi', 'pcpname',
       'ehn_vs_non', 'employed_vs_independent', 'ehn_practice', 'lhn',
       'entity', 'cost', 'paiddate', 'placeofservice',
       'placeofservicedescription', 'primary_cpt', 'cptdescription',
       'primary_icd', 'icddescription', 'servicing_prov',
       'servicing_specialty', 'rptgrouper', 'servicefacility',
       'servicefacilitynpi', 'readmission', 'readmission60', 'readmission90',
       'followup7dayind', 'followup30daytcoc', 'dischargedisposition'],
      dtype='object')

In [5]:
gt_ip.deid_personid.nunique()

16991

In [6]:
gt_ip.cost.describe()

count      24,206.00
mean       27,998.93
std        65,493.94
min        -5,935.80
25%         9,195.20
50%        15,542.66
75%        26,782.78
max     3,469,864.44
Name: cost, dtype: float64

In [7]:
#gt_ip["readmission"].value_counts(dropna=False, normalize=True)

In [9]:
# Parse dates
gt_ip["admitdate"] = pd.to_datetime(gt_ip["admitdate"], errors="coerce")
gt_ip["dischargedate"] = pd.to_datetime(gt_ip["dischargedate"], errors="coerce")

# LOS (min 1 day) 
los_raw = (gt_ip["dischargedate"] - gt_ip["admitdate"]).dt.days
gt_ip["los_days"] = los_raw

gt_ip["los_days"] = gt_ip["los_days"].clip(lower=1)

gt_ip[["admitdate", "dischargedate", "los_days"]].head(10)

,admitdate,dischargedate,los_days
0,2024-12-22,2024-12-29,7
1,2024-11-30,2024-12-13,13
2,2024-12-04,2024-12-16,12
3,2024-11-30,2024-12-04,4
4,2024-05-13,2024-05-16,3
5,2024-11-25,2024-12-02,7
6,2024-06-19,2024-06-21,2
7,2024-03-29,2024-04-03,5
8,2024-10-29,2024-11-08,10
9,2024-11-16,2024-11-19,3


In [10]:
gt_ip.to_csv("gt_ip_processed.csv", index=False)

### Dx (ICD frequency + cost)

In [11]:
#top 20 primary_icd codes
gt_ip.primary_icd.value_counts().head(20)

primary_icd
A41.9      1062
I13.0       435
N17.9       369
I11.0       347
O34.211     318
O48.0       303
I21.4       301
I48.0       219
U07.1       214
F33.2       203
J18.9       202
O13.4       199
F10.20      193
J44.1       178
N39.0       161
M48.062     157
A41.51      156
J96.01      149
I26.99      133
I25.110     130
Name: count, dtype: int64

In [12]:
# total costs for the top 20 primary_icd codes (from gt_ip) in Millions
top20_codes = gt_ip.primary_icd.value_counts().head(20).index
costs_top20 = (gt_ip[gt_ip.primary_icd.isin(top20_codes)]
               .groupby('primary_icd', as_index=False)['cost']
               .sum()
               .sort_values('cost', ascending=False))
costs_top20['cost'] = costs_top20['cost'] / 1e6  # Convert to Millions
costs_top20

,primary_icd,cost
1,A41.9,27.72
5,I13.0,12.30
6,I21.4,10.97
13,M48.062,8.68
17,O34.211,7.15
4,I11.0,6.44
18,O48.0,5.77
14,N17.9,5.53
12,J96.01,5.48
7,I25.110,5.47


In [13]:
# average cost per event (and per day) for the top 20 ICDs
ip_top20 = gt_ip[gt_ip.primary_icd.isin(top20_codes)].copy()
ip_top20['admitdate'] = pd.to_datetime(ip_top20['admitdate'])
ip_top20['dischargedate'] = pd.to_datetime(ip_top20['dischargedate'])
ip_top20['los_days'] = (ip_top20['dischargedate'] - ip_top20['admitdate']).dt.days.clip(lower=1)

result = (ip_top20.groupby('primary_icd', as_index=False)
          .agg(n_events=('cost', 'size'),
               avg_cost_per_event=('cost', 'mean'),
               total_cost=('cost', 'sum'),
               total_days=('los_days', 'sum')))

result['avg_cost_per_day'] = result['total_cost'] / result['total_days']
result = result.sort_values('avg_cost_per_event', ascending=False).reset_index(drop=True)
result

,primary_icd,n_events,avg_cost_per_event,total_cost,total_days,avg_cost_per_day
0,M48.062,157,"55,273.65","8,677,963.28",784,"11,068.83"
1,I25.110,130,"42,063.81","5,468,295.18",580,"9,428.10"
2,J96.01,149,"36,757.85","5,476,919.51",1292,"4,239.10"
3,I21.4,301,"36,442.92","10,969,317.52",1254,"8,747.46"
4,I13.0,435,"28,278.59","12,301,188.14",3130,"3,930.09"
5,A41.9,1062,"26,104.47","27,722,949.45",7311,"3,791.95"
6,A41.51,156,"23,876.30","3,724,702.69",1024,"3,637.40"
7,O34.211,318,"22,489.72","7,151,730.85",875,"8,173.41"
8,O13.4,199,"20,677.08","4,114,739.25",640,"6,429.28"
9,I26.99,133,"20,552.66","2,733,504.05",527,"5,186.91"


In [14]:
# average cost per event (and per day) for the top 20 ICDs (from icddescription)

result_dx = (ip_top20.groupby('icddescription', as_index=False)
               .agg(n_events=('cost', 'size'),
                    avg_cost_per_event=('cost', 'mean'),
                    total_cost=('cost', 'sum'),
                    total_days=('los_days', 'sum')))

result_dx['avg_cost_per_day'] = result_dx['total_cost'] / result_dx['total_days']
result_dx_desc = result_dx.sort_values('avg_cost_per_event', ascending=False).reset_index(drop=True)

result_dx_desc

,icddescription,n_events,avg_cost_per_event,total_cost,total_days,avg_cost_per_day
0,"Spinal stenosis, lumbar region with neurogenic...",157,"55,273.65","8,677,963.28",784,"11,068.83"
1,Atherosclerotic heart disease of native corona...,130,"42,063.81","5,468,295.18",580,"9,428.10"
2,Acute respiratory failure with hypoxia,149,"36,757.85","5,476,919.51",1292,"4,239.10"
3,Non-ST elevation (NSTEMI) myocardial infarction,301,"36,442.92","10,969,317.52",1254,"8,747.46"
4,Hypertensive heart and chronic kidney disease ...,435,"28,278.59","12,301,188.14",3130,"3,930.09"
5,"Sepsis, unspecified organism",1062,"26,104.47","27,722,949.45",7311,"3,791.95"
6,Sepsis due to Escherichia coli [E. coli],156,"23,876.30","3,724,702.69",1024,"3,637.40"
7,Maternal care for low transverse scar from pre...,318,"22,489.72","7,151,730.85",875,"8,173.41"
8,Gestational [pregnancy-induced] hypertension w...,199,"20,677.08","4,114,739.25",640,"6,429.28"
9,Other pulmonary embolism without acute cor pul...,133,"20,552.66","2,733,504.05",527,"5,186.91"


In [15]:
# top 20 diagnoses by volume + avg cost
top20_icd_volume = (
    gt_ip.groupby('icddescription', as_index=False)
    .agg(n_events=('cost', 'size'),
        avg_cost_per_event=('cost', 'mean'))
    .sort_values('n_events', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

top20_icd_volume

,icddescription,n_events,avg_cost_per_event
0,"Sepsis, unspecified organism",1062,"26,104.47"
1,Hypertensive heart and chronic kidney disease ...,435,"28,278.59"
2,"Acute kidney failure, unspecified",369,"14,978.34"
3,Hypertensive heart disease with heart failure,347,"18,556.27"
4,Maternal care for low transverse scar from pre...,318,"22,489.72"
5,Post-term pregnancy,303,"19,033.93"
6,Non-ST elevation (NSTEMI) myocardial infarction,301,"36,442.92"
7,Paroxysmal atrial fibrillation,219,"16,609.61"
8,COVID-19,214,"14,230.58"
9,"Major depressive disorder, recurrent severe wi...",203,"13,090.01"


## Cost drivers by payer, setting, network, follow-up, disposition

In [16]:
#planpayer
planpayer_cost = (
    gt_ip.groupby("planpayer", as_index=False)
    .agg(
        count=("cost", "size"),
        avg_cost=("cost", "mean"),
    )
    .sort_values("avg_cost", ascending=False)
    .reset_index(drop=True)
)
planpayer_cost

,planpayer,count,avg_cost
0,Emory Cigna Commercial,1057,"95,675.20"
1,Emory Anthem Commercial,7482,"36,914.74"
2,Emory United Healthcare Commercial,1086,"35,987.39"
3,Emory Aetna Commercial,1347,"29,855.18"
4,Emory Cigna MAdv,223,"19,289.82"
5,Emory Anthem MAdv,837,"18,417.06"
6,Emory MSSP,5135,"17,390.27"
7,Emory Aetna MAdv,2231,"17,273.31"
8,Emory Humana MAdv,2372,"15,630.60"
9,Emory United Healthcare MAdv Individual,1258,"15,201.70"


In [17]:
#place of service
pos_cost = (
    gt_ip.groupby("placeofservicedescription", as_index=False)
    .agg(
        n_events=("cost", "size"),
        avg_cost_per_event=("cost", "mean"),
        median_cost_per_event=("cost", "median"),
    )
    .sort_values("avg_cost_per_event", ascending=False)
    .reset_index(drop=True)
)
pos_cost

,placeofservicedescription,n_events,avg_cost_per_event,median_cost_per_event
0,Comprehensive Inpatient Rehabilitation Facility,14,"34,374.51","27,968.92"
1,Hospice,2,"30,460.72","30,460.72"
2,Inpatient Hospital,22103,"29,288.72","16,123.93"
3,Psychiatric Residential Treatment Center,39,"27,729.18","12,368.24"
4,Unknown,256,"20,053.86","12,546.33"
5,On Campus-Outpatient Hospital,278,"18,743.81","10,312.29"
6,Skilled Nursing Facility,1219,"13,134.33","10,325.32"
7,Home,6,"12,765.33","3,933.45"
8,Residential Substance Abuse Treatment Facility,187,"10,807.46","5,807.77"
9,Inpatient Psychiatric Facility,3,"6,570.10","6,290.42"


In [18]:
# follow-up 7 days
gt_ip.groupby("followup7dayind")["cost"].mean()

followup7dayind
0   27,832.32
1   28,349.83
Name: cost, dtype: float64

In [19]:
# Network and PCP employment
gt_ip.groupby("ehn_vs_non")["cost"].mean()

ehn_vs_non
EHN       27,451.39
Unknown   30,131.60
Name: cost, dtype: float64

In [20]:
gt_ip.groupby("employed_vs_independent")["cost"].mean()

employed_vs_independent
EHN Employed      24,224.54
EHN Independent   29,951.82
Unknown           30,131.60
Name: cost, dtype: float64

In [21]:
# Disposition
cost_by_disposition = (
    gt_ip.groupby("dischargedisposition", as_index=False)
    .agg(
        n_events=("cost", "size"),
        avg_cost_per_event=("cost", "mean"),
    )
    .sort_values("avg_cost_per_event", ascending=False)
    .reset_index(drop=True)
)
cost_by_disposition

,dischargedisposition,n_events,avg_cost_per_event
0,IP SURGICAL (PEDIATRIC),1,"288,277.00"
1,IP LTC/REHAB,532,"76,659.13"
2,IP MEDICAL (PEDIATRIC),4,"67,312.63"
3,IP OTHER,16,"49,465.37"
4,HOSPICE,27,"35,089.21"
5,IP MEDICAL (ADULT),302,"30,563.40"
6,HOME HEALTH,2199,"29,766.26"
7,HOME,19814,"26,792.04"
8,IP SURGICAL (ADULT),157,"24,448.65"
9,IP MATERNITY,7,"24,182.93"


## Specialty & facility drill-down

In [22]:
# Specialty: cost + LOS + facility_count
top20_specialty = (
    gt_ip.groupby("servicing_specialty", as_index=False)
    .agg(
        mean_cost=("cost", "mean"),
        mean_los=("los_days", "mean"),
        facility_count=("servicefacilitynpi", "nunique"),
    )
    .sort_values("mean_cost", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20_specialty

,servicing_specialty,mean_cost,mean_los,facility_count
0,Pediatric Critical Care Medicine,"379,471.74",21.30,7
1,Long Term Care Hospital,"234,880.82",52.00,1
2,Counselor,"171,323.42",27.00,3
3,Thoracic Surgery (Cardiothoracic Vascular Surg...,"103,021.17",7.54,47
4,Medical Oncology,"100,057.63",8.35,11
5,Transplant Hepatology,"77,065.73",12.11,2
6,Neurological Surgery,"76,443.67",7.17,65
7,Otolaryngology,"72,206.34",6.87,25
8,Hepatology,"65,709.60",19.00,2
9,Pediatric Cardiology,"62,120.92",4.83,7


In [23]:
gt_ip["servicefacilitynpi"].nunique()

936

## Readmission costs

In [24]:
gt_ip.groupby("readmission")["cost"].mean() #30days

readmission
0   27,952.67
1   28,568.87
Name: cost, dtype: float64

In [25]:
gt_ip.groupby("readmission60")["cost"].mean() #60 days

readmission60
0   28,113.75
1   27,107.44
Name: cost, dtype: float64

In [26]:
gt_ip.groupby("readmission90")["cost"].mean() #90 days

readmission90
0   28,238.72
1   26,506.05
Name: cost, dtype: float64

In [27]:
#Dx with most readmissions

top20_readmit_dx = (
    gt_ip.loc[gt_ip["readmission"] == 1]
    .groupby("icddescription", as_index=False)
    .agg(
        n_readmissions=("readmission", "size"),
        avg_cost_readmit_event=("cost", "mean"),
    )
    .sort_values("n_readmissions", ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top20_readmit_dx

,icddescription,n_readmissions,avg_cost_readmit_event
0,"Sepsis, unspecified organism",96,"22,387.73"
1,Hypertensive heart and chronic kidney disease ...,83,"56,104.34"
2,"Hb-SS disease with crisis, unspecified",48,"16,566.30"
3,"Acute kidney failure, unspecified",48,"16,670.50"
4,Hypertensive heart disease with heart failure,38,"15,073.12"
5,"Alcohol dependence, uncomplicated",30,"12,809.34"
6,Chronic obstructive pulmonary disease with (ac...,22,"9,549.32"
7,"Major depressive disorder, recurrent severe wi...",21,"13,855.27"
8,"Schizoaffective disorder, bipolar type",19,"10,178.49"
9,Hypertensive heart and chronic kidney disease ...,18,"17,639.87"


## Patient utilization (duplicates / multiple encounters)

In [28]:
gt_ip.deid_personid.nunique()

16991

In [29]:
# patients with >1 encounter, number of encounters and total cost
patient_cost_summary = (
    gt_ip.groupby('deid_personid', as_index=False)
    .agg(n_visits=('deid_personid', 'size'),
        total_cost=('cost', 'sum'))
    .sort_values('total_cost', ascending=False)
    .reset_index(drop=True))

patients_with_multiple_visits = (
    patient_cost_summary
    .query("n_visits > 1")
    .sort_values(['total_cost', 'n_visits'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True))

patients_with_multiple_visits

,deid_personid,n_visits,total_cost
0,2139a0e83f011f14e40d27d9c36cd306,6,"4,506,790.05"
1,c5a2c32292a2db700ce9b9a5308fe78a,6,"2,861,004.04"
2,b9ecb4ada96ac400e4eab2ef89e3b561,2,"2,809,100.77"
3,e90818bd9b66e053bbe53db794ba0669,2,"2,486,941.82"
4,0755fe99f76098018969ec6feca34836,9,"2,429,074.41"
5,a4185eac1990a4d6fcacb30037fe82f2,9,"2,145,860.94"
6,7b9826013f247071f76f3eedd6f7b82a,7,"2,019,984.15"
7,2081290eae222e68451c2be8cb40761a,5,"1,683,800.86"
8,971d8072c795527e57fc987b8c161525,2,"1,201,059.90"
9,cdaabb51a0ea0c54648e03df7f3b08d0,2,"1,106,414.45"


In [30]:
# patients with >1 encounter, number of encounters and total cost
patients_with_multiple_visits = (
    patient_cost_summary.query("n_visits > 1")
    .sort_values(["total_cost", "n_visits"], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)
patients_with_multiple_visits

,deid_personid,n_visits,total_cost
0,2139a0e83f011f14e40d27d9c36cd306,6,"4,506,790.05"
1,c5a2c32292a2db700ce9b9a5308fe78a,6,"2,861,004.04"
2,b9ecb4ada96ac400e4eab2ef89e3b561,2,"2,809,100.77"
3,e90818bd9b66e053bbe53db794ba0669,2,"2,486,941.82"
4,0755fe99f76098018969ec6feca34836,9,"2,429,074.41"
5,a4185eac1990a4d6fcacb30037fe82f2,9,"2,145,860.94"
6,7b9826013f247071f76f3eedd6f7b82a,7,"2,019,984.15"
7,2081290eae222e68451c2be8cb40761a,5,"1,683,800.86"
8,971d8072c795527e57fc987b8c161525,2,"1,201,059.90"
9,cdaabb51a0ea0c54648e03df7f3b08d0,2,"1,106,414.45"


In [31]:
# top 10 encounters
top_patients_by_visits = (
    patient_cost_summary.sort_values(["n_visits", "total_cost"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)
top_patients_by_visits

,deid_personid,n_visits,total_cost
0,52a8219780d9b8afb79a176200d5566a,24,"295,242.58"
1,f5a24541d4a9f2a73b81187b9bc5a2c0,23,"237,995.45"
2,07d8e1aebcf375fcec98d06651a07157,20,"207,141.26"
3,3789dce2f4534a970f241857c923fb37,17,"172,770.58"
4,b06cd7d5ded46b72754e1e462efc5275,14,"147,513.29"
5,100943f5f753077ab124feb62eebbfab,13,"181,700.00"
6,54e0df5cf00ac35fdf775998d1b0077d,12,"189,294.98"
7,2cf1ef18232944481c58fd31a4f26737,12,"134,074.82"
8,2e0a8beaf11303d18d8b0307634bda22,12,"19,984.45"
9,7f04535f71d93323d7b433d344cdc934,11,"559,442.49"


In [32]:
# top 10 by visits + costliest diagnosis per patient
costliest_dx_per_patient = (
    gt_ip.groupby(["deid_personid", "icddescription"], as_index=False)
    .agg(total_dx_cost=("cost", "sum"))
    .sort_values(["deid_personid", "total_dx_cost"], ascending=[True, False])
    .drop_duplicates(subset="deid_personid")
    .rename(columns={"icddescription": "costliest_dx"})
    [["deid_personid", "costliest_dx"]])

top_patients_by_visits_with_dx = (
    patient_cost_summary.merge(costliest_dx_per_patient, on="deid_personid", how="left")
    .sort_values(["n_visits", "total_cost"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True))
top_patients_by_visits_with_dx

,deid_personid,n_visits,total_cost,costliest_dx
0,52a8219780d9b8afb79a176200d5566a,24,"295,242.58",Alcohol induced acute pancreatitis without nec...
1,f5a24541d4a9f2a73b81187b9bc5a2c0,23,"237,995.45","Hb-SS disease with crisis, unspecified"
2,07d8e1aebcf375fcec98d06651a07157,20,"207,141.26","Hb-SS disease with crisis, unspecified"
3,3789dce2f4534a970f241857c923fb37,17,"172,770.58","Major depressive disorder, recurrent severe wi..."
4,b06cd7d5ded46b72754e1e462efc5275,14,"147,513.29",Hypertensive heart and chronic kidney disease ...
5,100943f5f753077ab124feb62eebbfab,13,"181,700.00",Other cerebral infarction
6,54e0df5cf00ac35fdf775998d1b0077d,12,"189,294.98",Metabolic encephalopathy
7,2cf1ef18232944481c58fd31a4f26737,12,"134,074.82",Infection and inflammatory reaction due to ind...
8,2e0a8beaf11303d18d8b0307634bda22,12,"19,984.45",Other fluid overload
9,7f04535f71d93323d7b433d344cdc934,11,"559,442.49",Bloodstream infection due to central venous ca...


In [33]:
# correlation n_visits vs total_cost
visit_cost_correlation = patient_cost_summary[["n_visits", "total_cost"]].corr().iloc[0, 1]
print("Correlation between visits and total cost:", visit_cost_correlation)

Correlation between visits and total cost: 0.3020681553920538


High-cost utilization appears to be driven more by acute decompensations of chronic disease and severe infections than by visit frequency alone.

## High-cost + multi-readmit cohort case

In [34]:
patient_summary = (
    gt_ip.groupby("deid_personid", as_index=False)
    .agg(
        n_visits=("deid_personid", "size"),
        total_cost=("cost", "sum"),
        n_readmit_30=("readmission", "sum"),
        n_readmit_60=("readmission60", "sum"),
        n_readmit_90=("readmission90", "sum"),
    )
)

high_cost_threshold = patient_summary["total_cost"].quantile(0.95)

high_cost_multi_readmit_patients = (
    patient_summary.query("total_cost >= @high_cost_threshold and n_readmit_30 > 1")
    .sort_values(["total_cost", "n_readmit_30"], ascending=[False, False])
    .reset_index(drop=True)
)

high_cost_multi_readmit_patients.head(20)

,deid_personid,n_visits,total_cost,n_readmit_30,n_readmit_60,n_readmit_90
0,0755fe99f76098018969ec6feca34836,9,"2,429,074.41",4,7,7
1,a4185eac1990a4d6fcacb30037fe82f2,9,"2,145,860.94",3,4,5
2,7b9826013f247071f76f3eedd6f7b82a,7,"2,019,984.15",4,5,5
3,1fa73aad2c91c261daa3172bc77d0bf3,7,"875,620.20",3,3,3
4,cbdb9f81a1478de81ef9067c50d78235,9,"847,150.46",3,3,4
5,48e6eb4c2be000e84659e5eaa8f89227,6,"832,013.24",5,5,5
6,ccd9da86e896ee96b1aa0e4ade9677a0,4,"822,001.78",2,2,2
7,5a77b63b65984b29dac37c829a242750,7,"817,286.56",2,4,4
8,e57fe488e875e20acda18b3d4147613e,6,"791,381.77",3,3,3
9,cfa24cf832186620e57de64aeaf5c96c,4,"613,312.16",2,3,3


In [35]:
# encounters in cohort
high_cost_ids = high_cost_multi_readmit_patients["deid_personid"]

high_cost_multi_readmit_encounters = gt_ip.loc[gt_ip["deid_personid"].isin(high_cost_ids)].copy()

In [36]:
# top dx in cohort

top20_icd_high_cost_multi_readmit = (
    high_cost_multi_readmit_encounters.groupby("icddescription", as_index=False)
    .agg(
        n_events=("cost", "size"),
        total_cost=("cost", "sum"),
        avg_cost_per_event=("cost", "mean"),
        n_unique_patients=("deid_personid", "nunique"),
    )
    .sort_values(["total_cost", "n_events"], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)
top20_icd_high_cost_multi_readmit

,icddescription,n_events,total_cost,avg_cost_per_event,n_unique_patients
0,Hypertensive heart and chronic kidney disease ...,39,"3,363,311.99","86,238.77",10
1,Infection and inflammatory reaction due to oth...,5,"1,954,245.84","390,849.17",1
2,"Sepsis, unspecified organism",29,"1,229,570.15","42,398.97",21
3,Dehiscence of amputation stump,1,"826,792.94","826,792.94",1
4,Bloodstream infection due to central venous ca...,13,"777,336.79","59,795.14",5
5,"Hb-SS disease with crisis, unspecified",47,"771,424.91","16,413.30",4
6,Alcoholic cirrhosis of liver with ascites,10,"720,987.01","72,098.70",4
7,Acute on chronic combined systolic (congestive...,1,"698,009.75","698,009.75",1
8,Sepsis due to Methicillin resistant Staphyloco...,3,"610,248.51","203,416.17",3
9,Type 2 diabetes mellitus with diabetic periphe...,5,"572,978.87","114,595.77",3


In [37]:
# diagnoses only within readmission encounters

readmission_encounters = high_cost_multi_readmit_encounters.query("readmission == 1")

top20_icd_in_readmissions = (
    readmission_encounters.groupby("icddescription", as_index=False)
    .agg(
        n_events=("cost", "size"),
        total_cost=("cost", "sum"),
        avg_cost_per_event=("cost", "mean"),
        n_unique_patients=("deid_personid", "nunique"),
    )
    .sort_values(["total_cost", "n_events"], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)
top20_icd_in_readmissions

,icddescription,n_events,total_cost,avg_cost_per_event,n_unique_patients
0,Hypertensive heart and chronic kidney disease ...,25,"2,957,744.53","118,309.78",9
1,Dehiscence of amputation stump,1,"826,792.94","826,792.94",1
2,"Sepsis, unspecified organism",19,"760,332.53","40,017.50",15
3,"Hb-SS disease with crisis, unspecified",37,"614,473.05","16,607.38",3
4,Sepsis due to Methicillin resistant Staphyloco...,2,"605,333.59","302,666.80",2
5,Other specified diseases of upper respiratory ...,3,"446,471.54","148,823.85",1
6,Angiodysplasia of stomach and duodenum with bl...,3,"317,686.31","105,895.44",3
7,"Acute myeloblastic leukemia, not having achiev...",1,"312,784.62","312,784.62",1
8,"Infection following a procedure, organ and spa...",4,"293,213.63","73,303.41",3
9,Toxic gastroenteritis and colitis,1,"287,003.54","287,003.54",1


In [38]:
## Typical Pt. clinical profile

clinical_profile = (
    high_cost_multi_readmit_patients[["n_visits", "total_cost", "n_readmit_30"]]
    .agg(["mean", "median", "min", "max"])
)
clinical_profile

,n_visits,total_cost,n_readmit_30
mean,7.35,"358,530.92",3.64
median,6.50,"239,396.71",3.00
min,3.00,"125,408.92",2.00
max,24.00,"2,429,074.41",22.00


In [39]:
# top specialties + top dx by unique patients

top5_specialties = (
    high_cost_multi_readmit_encounters.groupby("servicing_specialty", as_index=False)
    .agg(
        n_events=("cost", "size"),
        n_patients=("deid_personid", "nunique"),
    )
    .sort_values("n_patients", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
top5_specialties

,servicing_specialty,n_events,n_patients
0,Hospitalist,120,46
1,Internal Medicine,87,42
2,Emergency Medicine,54,26
3,Nurse Practitioner,44,24
4,Radiology,24,19


High-cost multi-readmit patients are primarily managed by general services rather than specialty or procedural departments. Spending is concentrated in patients with acute decompensations.

In [40]:
top5_dx = (
    high_cost_multi_readmit_encounters.groupby("icddescription", as_index=False)
    .agg(
        n_events=("cost", "size"),
        n_patients=("deid_personid", "nunique"),
    )
    .sort_values("n_patients", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
top5_dx

,icddescription,n_events,n_patients
0,"Sepsis, unspecified organism",29,21
1,Hypertensive heart and chronic kidney disease ...,39,10
2,"Pneumonia, unspecified organism",8,7
3,"Acute kidney failure, unspecified",7,7
4,Acute pancreatitis without necrosis or infecti...,13,6


High-cost multi-readmit patients in this population are primarily driven by severe systemic and organ-failure conditions, suggesting that cost concentration is driven by clinical instability rather than elective or procedural care. Potential opportunities for targeted post-discharge monitoring and chronic disease management (home monitoring for CHF/CKD, early sepsis detection, etc.)

## Key findings

- The highest-cost diagnoses by total spend and case volume follow expected epidemiological patterns, with chronic conditions, infections, and complex systemic diseases dominating inpatient utilization. https://doi.org/10.1016/j.mayocpiqo.2023.08.005, https://jamanetwork.com/journals/jama/fullarticle/2830568
- Cost concentration is strongly associated with settings involving longer lengths of stay (rehabilitation, hospice, and high-acuity inpatient units)
- The most expensive conditions by average cost per event tend to correspond to rare or late-stage diseases, as well as atypical procedures and severe complications.
- Patients with 30-day readmissions incur substantially higher total costs, suggesting that clinical instability and chronic disease progression are major cost drivers.
- The high-cost, multi-readmission cohort is primarily characterized by severe systemic and organ-failure conditions.
- Specialty and facility analyses suggest that high-acuity services disproportionately contribute to overall spending.
